In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [2]:
df = pd.read_csv("insurance.csv")

print("Initial shape:", df.shape)

Initial shape: (1338, 7)


In [3]:
df_drop = df.dropna()

In [4]:
df_fill = df.copy()

# Fill numeric with mean
for col in df_fill.select_dtypes(include=['int64', 'float64']).columns:
    df_fill[col] = df_fill[col].fillna(df_fill[col].mean())

# Fill categorical with mode
for col in df_fill.select_dtypes(include=['object']).columns:
    df_fill[col] = df_fill[col].fillna(df_fill[col].mode()[0])



C:\Users\kmfur\AppData\Local\Temp\ipykernel_7948\575863140.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_fill.select_dtypes(include=['object']).columns:


In [5]:
def remove_outliers(data):
    df_out = data.copy()
    
    for col in df_out.select_dtypes(include=['int64', 'float64']).columns:
        Q1 = df_out[col].quantile(0.25)
        Q3 = df_out[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df_out = df_out[(df_out[col] >= lower) & (df_out[col] <= upper)]
    
    return df_out

df_drop = remove_outliers(df_drop)
df_fill = remove_outliers(df_fill)

In [6]:
df_drop = pd.get_dummies(df_drop, drop_first=True)
df_fill = pd.get_dummies(df_fill, drop_first=True)

In [12]:
def split_scale(df_data):
    X = df_data.drop("charges", axis=1)
    y = df_data["charges"]
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test

X1_train, X1_test, y1_train, y1_test = split_scale(df_drop)
X2_train, X2_test, y2_train, y2_test = split_scale(df_fill)


In [13]:
def evaluate_model(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"\n--- {label} ---")
    print("RMSE:", rmse)
    print("R² Score:", r2)

In [14]:
lr = LinearRegression()
lr.fit(X1_train, y1_train)
pred_lr_drop = lr.predict(X1_test)
evaluate_model(y1_test, pred_lr_drop, "Linear Regression (Drop Missing)")

lr.fit(X2_train, y2_train)
pred_lr_fill = lr.predict(X2_test)
evaluate_model(y2_test, pred_lr_fill, "Linear Regression (Fill Missing)")



--- Linear Regression (Drop Missing) ---
RMSE: 4458.441052676729
R² Score: 0.6328135163446322

--- Linear Regression (Fill Missing) ---
RMSE: 4458.441052676729
R² Score: 0.6328135163446322


In [15]:
poly = PolynomialFeatures(degree=2)
X1_train_poly = poly.fit_transform(X1_train)
X1_test_poly = poly.transform(X1_test)
X2_train_poly = poly.fit_transform(X2_train)
X2_test_poly = poly.transform(X2_test)

poly_lr = LinearRegression()
poly_lr.fit(X1_train_poly, y1_train)
pred_poly_drop = poly_lr.predict(X1_test_poly)
evaluate_model(y1_test, pred_poly_drop, "Polynomial Regression (Drop Missing)")

poly_lr.fit(X2_train_poly, y2_train)
pred_poly_fill = poly_lr.predict(X2_test_poly)
evaluate_model(y2_test, pred_poly_fill, "Polynomial Regression (Fill Missing)")


--- Polynomial Regression (Drop Missing) ---
RMSE: 4231.132751042885
R² Score: 0.6693002008570255

--- Polynomial Regression (Fill Missing) ---
RMSE: 4231.132751042885
R² Score: 0.6693002008570255


In [16]:
rf1 = RandomForestRegressor(n_estimators=100, max_depth=5, max_features='sqrt', random_state=42)
rf1.fit(X1_train, y1_train)
pred_rf_drop = rf1.predict(X1_test)
evaluate_model(y1_test, pred_rf_drop, "Random Forest (Drop Missing)")

rf2 = RandomForestRegressor(n_estimators=200, max_depth=10, max_features='log2', random_state=42)
rf2.fit(X2_train, y2_train)
pred_rf_fill = rf2.predict(X2_test)
evaluate_model(y2_test, pred_rf_fill, "Random Forest (Fill Missing)")


--- Random Forest (Drop Missing) ---
RMSE: 4720.435335490297
R² Score: 0.5883911219265396

--- Random Forest (Fill Missing) ---
RMSE: 4374.497276612529
R² Score: 0.6465101653886078
